In [2]:
import pandas as pd
import numpy as np
import csv
from collections import defaultdict

def load_wand_points_gui_style(file_path):
    """
    参考 WandCalibrator.load_wand_data_from_csv 和 
    RefractiveWandCalibrator._collect_observations 的逻辑进行读取
    """
    wand_points = defaultdict(lambda: defaultdict(list))
    
    with open(file_path, 'r', newline='') as f:
        reader = csv.reader(f)
        header = next(reader, None)
        if not header:
            print("Empty file")
            return None
            
        # Header 顺序: ["Frame", "Camera", "Status", "PointIdx", "X", "Y", "Radius", "Metric"]
        for row in reader:
            if len(row) < 8: continue
            
            try:
                f_idx = int(row[0])
                c_idx = int(row[1])
                status = row[2]
                pt_idx = int(row[3])
                x = float(row[4])
                y = float(row[5])
                r = float(row[6])
                m = float(row[7])
                
                # 仅加载 Filtered 状态的点（排除 Raw 点，模仿重构校准逻辑）
                if status.startswith("Filtered"):
                    wand_points[f_idx][c_idx].append([x, y, r, m, status, pt_idx])
            except (ValueError, IndexError): 
                continue

    # 将数据整理为 observations 结构 (模仿 _collect_observations)
    frames = sorted(wand_points.keys())
    # 获取所有涉及到的 Camera ID
    cam_ids = sorted(list(set(cid for f in wand_points.values() for cid in f.keys())))
    
    obsA = defaultdict(dict)       # obsA[frame][cam] = [x, y]
    obsB = defaultdict(dict)       # obsB[frame][cam] = [x, y]
    radii_small = defaultdict(dict) # radii_small[frame][cam] = r
    radii_large = defaultdict(dict) # radii_large[frame][cam] = r
    
    for fid in frames:
        for cid in cam_ids:
            pts = wand_points[fid].get(cid, [])
            for pt in pts:
                x, y, r, m, status, pt_idx = pt
                if status == "Filtered_Small":
                    obsA[fid][cid] = [x, y]
                    radii_small[fid][cid] = r
                elif status == "Filtered_Large":
                    obsB[fid][cid] = [x, y]
                    radii_large[fid][cid] = r
                    
    print(f"Loaded {len(frames)} frames for cams: {cam_ids}")
    return {
        "frames": frames,
        "cam_ids": cam_ids,
        "obsA": dict(obsA),
        "obsB": dict(obsB),
        "radii_small": dict(radii_small),
        "radii_large": dict(radii_large)
    }

# 路径设置
wand_points_path = r"H:\20260106\T0\wand_points3_filtered.csv"
dataset = load_wand_points_gui_style(wand_points_path)

# 打印第一帧样本进行验证
if dataset and dataset['frames']:
    fid0 = dataset['frames'][0]
    print(f"\nSample Frame {fid0} Data Summary:")
    for cid in dataset['cam_ids']:
        has_A = cid in dataset['obsA'].get(fid0, {})
        has_B = cid in dataset['obsB'].get(fid0, {})
        if has_A or has_B:
            info_A = f"Small={dataset['obsA'][fid0][cid]}" if has_A else "Small=N/A"
            info_B = f"Large={dataset['obsB'][fid0][cid]}" if has_B else "Large=N/A"
            print(f"  Cam {cid}: {info_A}, {info_B}")

Loaded 2331 frames for cams: [0, 1, 2]

Sample Frame 0 Data Summary:
  Cam 0: Small=[674.402, 489.376], Large=[824.829, 771.021]
  Cam 1: Small=[281.742, 482.818], Large=[592.633, 735.946]
  Cam 2: Small=[260.508, 418.651], Large=[573.791, 611.049]


In [4]:
import os
import openlpt as lpt

# 路径设置：指向包含 cam0.txt, cam1.txt, cam2.txt 的目录
cam_dir = r"H:\20260106\T0\Refraction\Calibration1\camFile"
cams = {}

# 加载相机 0, 1, 2
for cid in [0, 1, 2]:
    cam_path = os.path.join(cam_dir, f"cam{cid}.txt")
    
    if os.path.exists(cam_path):
        # 创建 lpt Camera 对象
        cam = lpt.Camera()
        # 加载参数
        cam.loadParameters(cam_path)
        
        cams[cid] = cam
        print(f"Successfully loaded Cam {cid} from {cam_path}")
        print(f"  Type: {cam._type}")
        print(f"  Resolution: {cam.getNCol()}x{cam.getNRow()}")
    else:
        print(f"Error: Camera file not found: {cam_path}")

# 验证 cams 字典
print(f"\nTotal loaded cameras: {len(cams)}")

Successfully loaded Cam 0 from H:\20260106\T0\Refraction\Calibration1\camFile\cam0.txt
  Type: CameraType.PINPLATE
  Resolution: 1280x800
Successfully loaded Cam 1 from H:\20260106\T0\Refraction\Calibration1\camFile\cam1.txt
  Type: CameraType.PINPLATE
  Resolution: 1280x800
Successfully loaded Cam 2 from H:\20260106\T0\Refraction\Calibration1\camFile\cam2.txt
  Type: CameraType.PINPLATE
  Resolution: 1280x800

Total loaded cameras: 3


In [45]:
import sys
import numpy as np

# 确保能导入项目内部模块
if '.' not in sys.path:
    sys.path.append('.')

from modules.camera_calibration.wand_calibration.refractive_geometry import (
    build_pinplate_ray_cpp, triangulate_point, point_to_ray_dist
)

def reconstruct_and_analyze(fid, dataset, cams):
    """重构 A, B 点并输出重构误差（射线到 3D 点的距离）"""
    obs_info = [("Point A", dataset['obsA'].get(fid, {})), 
                ("Point B", dataset['obsB'].get(fid, {}))]
    
    results_3d = []
    
    print(f"--- Frame {fid} Reconstruction Analysis ---")
    
    for label, obs in obs_info:
        if not obs:
            results_3d.append(None)
            continue
            
        # 1. 建立折射射线
        rays = []
        for cid, uv in obs.items():
            if cid in cams:
                ray = build_pinplate_ray_cpp(cams[cid], uv)
                if ray.valid: rays.append(ray)
        
        if len(rays) < 2:
            print(f"  {label}: 重构失败（有效射线不足）")
            results_3d.append(None)
            continue
            
        # 2. 三角重构
        XA_np, cond, success, msg = triangulate_point(rays)
        
        # 3. 计算重构误差 (Point-to-Ray Distance)
        # 单位通常是 mm (取决于相机参数中的 t_vec 单位)
        recon_errors = [point_to_ray_dist(XA_np, r.o, r.d) for r in rays]
        mean_recon_err = np.mean(recon_errors)
        
        print(f"\n  {label}:")
        print(f"    3D Position: {XA_np}")
        print(f"    Ray Errors (mm): {['{:.4f}'.format(e) for e in recon_errors]}")
        print(f"    Mean Recon Error: {mean_recon_err:.6f} mm")
        
        results_3d.append(XA_np)
        
    return results_3d[0], results_3d[1]

# 执行并保存 3D 坐标供下一个 cell 使用
if dataset['frames']:
    test_fid = dataset['frames'][0]
    XA_3d, XB_3d = reconstruct_and_analyze(test_fid, dataset, cams)
    
    if XA_3d is not None and XB_3d is not None:
        L = np.linalg.norm(XA_3d - XB_3d)
        print(f"\nReconstructed Wand Length: {L:.4f} mm")


--- Frame 0 Reconstruction Analysis ---

  Point A:
    3D Position: [-0.94835303  1.33830623  7.09354285]
    Ray Errors (mm): ['0.0079', '0.0696', '0.0711']
    Mean Recon Error: 0.049537 mm

  Point B:
    3D Position: [4.24164427 6.48229889 0.30211409]
    Ray Errors (mm): ['0.0405', '0.0924', '0.0584']
    Mean Recon Error: 0.063755 mm

Reconstructed Wand Length: 9.9760 mm


In [46]:
import openlpt as lpt

def verify_projection(X_3d_np, obs_dict, cams, label):
    if X_3d_np is None:
        print(f"{label} is None, skipping verification.")
        return

    print(f"\n--- {label} Reprojection Analysis ---")
    pt_world = lpt.Pt3D(float(X_3d_np[0]), float(X_3d_np[1]), float(X_3d_np[2]))
    
    total_err = 0
    count = 0
    
    for cid, uv_obs in obs_dict.items():
        if cid in cams:
            # 使用 C++project 函数
            uv_proj_pt = cams[cid].project(pt_world)
            uv_proj = np.array([uv_proj_pt[0], uv_proj_pt[1]])
            
            err = np.linalg.norm(uv_proj - np.array(uv_obs))
            total_err += err
            count += 1
            print(f"  Cam {cid}: Obs={uv_obs}, Proj=[{uv_proj[0]:.3f}, {uv_proj[1]:.3f}], Err={err:.4f} px")
            
    if count > 0:
        print(f"  Mean Error: {total_err/count:.6f} px")

# 调用验证逻辑
if dataset['frames']:
    fid = dataset['frames'][0]
    verify_projection(XA_3d, dataset['obsA'][fid], cams, "Point A")
    verify_projection(XB_3d, dataset['obsB'][fid], cams, "Point B")


--- Point A Reprojection Analysis ---
  Cam 0: Obs=[674.402, 489.376], Proj=[674.059, 489.469], Err=0.3557 px
  Cam 1: Obs=[281.742, 482.818], Proj=[278.993, 483.897], Err=2.9530 px
  Cam 2: Obs=[260.508, 418.651], Proj=[263.229, 417.490], Err=2.9585 px
  Mean Error: 2.089073 px

--- Point B Reprojection Analysis ---
  Cam 0: Obs=[824.829, 771.021], Proj=[825.494, 772.813], Err=1.9117 px
  Cam 1: Obs=[592.633, 735.946], Proj=[591.725, 732.143], Err=3.9104 px
  Cam 2: Obs=[573.791, 611.049], Proj=[574.651, 613.291], Err=2.4010 px
  Mean Error: 2.741035 px


In [47]:
import numpy as np
from modules.camera_calibration.wand_calibration.refractive_geometry import (
    build_pinplate_ray_cpp, triangulate_point, point_to_ray_dist
)
import openlpt as lpt

def project_and_reconstruct_check(X_3d_orig, cams, active_cids, label="Point"):
    if X_3d_orig is None:
        print(f"[{label}] Original 3D point is None. Skipping.")
        return

    print(f"\n=== {label} Consistency Check (3D -> 2D -> 3D) ===")
    print(f"  Original 3D: {X_3d_orig}")
    
    # 1. Project 3D -> 2D (Generate 'Perfect' Observations)
    proj_obs = {}
    pt_world_obj = lpt.Pt3D(float(X_3d_orig[0]), float(X_3d_orig[1]), float(X_3d_orig[2]))
    
    print("  [Step 1] Projected 2D Coords (Ideal):")
    for cid in active_cids:
        if cid in cams:
            uv = cams[cid].project(pt_world_obj)
            # Ensure list format [x, y] for verification
            proj_obs[cid] = [uv[0], uv[1]] 
            print(f"    Cam {cid}: {proj_obs[cid]}")

    # 2. Reconstruct 2D -> 3D (Refractive Triangulation)
    rays = []
    for cid, uv in proj_obs.items():
        # Build refractive ray from the PROJECTED coordinate
        ray = build_pinplate_ray_cpp(cams[cid], uv)
        if ray.valid:
            rays.append(ray)
    
    if len(rays) < 2:
        print("  [Error] Failed to build valid rays from projections.")
        return

    # Triangulate
    X_recon, cond, success, msg = triangulate_point(rays)
    
    if not success:
        print(f"  [Error] Triangulation failed: {msg}")
        return

    # 3. Compare & Metrics
    diff = X_recon - X_3d_orig
    dist_err = np.linalg.norm(diff)
    
    # Calculate Ray Errors (Self-consistency)
    recon_ray_errors = [point_to_ray_dist(X_recon, r.o, r.d) for r in rays]
    mean_ray_err = np.mean(recon_ray_errors)

    print("  [Step 2] Reconstructed 3D from Projections:")
    print(f"    New 3D: {X_recon}")
    print(f"    Diff (New - Orig): {diff}")
    print(f"    Euclidean Error: {dist_err:.6e} mm")
    print(f"    Mean Ray Discrepancy: {mean_ray_err:.6e} mm")
    
    if dist_err < 1e-4:
        print("  [RESULT] PASS (High Consistency)")
    else:
        print("  [RESULT] WARNING (Possible Refractive Logic Mismatch)")

# --- Run Check ---
if 'dataset' in locals() and dataset['frames']:
    # Get active camera IDs from the first frame's observations
    fid0 = dataset['frames'][0]
    
    # Check Point A
    active_cids_A = list(dataset['obsA'][fid0].keys())
    if 'XA_3d' in locals():
        project_and_reconstruct_check(XA_3d, cams, active_cids_A, "Point A")
        
    # Check Point B
    active_cids_B = list(dataset['obsB'][fid0].keys())
    if 'XB_3d' in locals():
        project_and_reconstruct_check(XB_3d, cams, active_cids_B, "Point B")


=== Point A Consistency Check (3D -> 2D -> 3D) ===
  Original 3D: [-0.94835303  1.33830623  7.09354285]
  [Step 1] Projected 2D Coords (Ideal):
    Cam 0: [674.0585632478679, 489.46870190849614]
    Cam 1: [278.99302434538896, 483.8965646571962]
    Cam 2: [263.22926109452834, 417.49020584415143]
  [Step 2] Reconstructed 3D from Projections:
    New 3D: [-0.94835152  1.33830515  7.09354391]
    Diff (New - Orig): [ 1.51003820e-06 -1.08870640e-06  1.06782513e-06]
    Euclidean Error: 2.146101e-06 mm
    Mean Ray Discrepancy: 1.238474e-06 mm
  [RESULT] PASS (High Consistency)

=== Point B Consistency Check (3D -> 2D -> 3D) ===
  Original 3D: [4.24164427 6.48229889 0.30211409]
  [Step 1] Projected 2D Coords (Ideal):
    Cam 0: [825.4941906121587, 772.8132824577385]
    Cam 1: [591.7246359865414, 732.1425623276585]
    Cam 2: [574.650629304337, 613.2907930369938]
  [Step 2] Reconstructed 3D from Projections:
    New 3D: [4.24164628 6.48229745 0.30211513]
    Diff (New - Orig): [ 2.0107861

In [6]:
def mtx_to_np(mtx):
    """手动将 MatrixDouble 转换为 numpy 数组"""
    rows = mtx.getDimRow()
    cols = mtx.getDimCol()
    res = np.zeros((rows, cols))
    for i in range(rows):
        for j in range(cols):
            # 使用 [i, j] 索引访问 (MatrixDouble 已绑定 __getitem__)
            res[i, j] = mtx[i, j]
    return res


def check_scale_and_magnification(XA_3d, cams):
    print("--- Scale & Magnification Analysis ---")
    for cid, cam in cams.items():
        # [CPP_PROTOCOL] 对于 PINPLATE 相机，直接访问 _pinplate_param
        # 它包含了继承自 PinholeParam 的 cam_mtx, r_mtx, t_vec
        if hasattr(cam, '_pinplate_param') and cam._type == lpt.CameraType.PINPLATE:
            pp = cam._pinplate_param
        else:
            pp = cam._pinhole_param
        
        # 1. 获取内参 f (cam_mtx[0,0])
        f = pp.cam_mtx[0, 0]
        
        # 2. 获取旋转矩阵 R 和位移向量 t (手动转 numpy)
        R = mtx_to_np(pp.r_mtx)
        t = np.array([pp.t_vec[i] for i in range(3)])
        
        # 3. 计算相机光心 C = -R^T * t
        C = -R.T @ t
        
        # 4. 计算 3D 点到相机中心的物理距离 (mm)
        dist_mm = np.linalg.norm(XA_3d - C)
        
        # 5. 比例：像素/毫米 (Magnification)
        px_per_mm = f / dist_mm
        
        print(f"  Cam {cid}:")
        print(f"    Focal Length: {f:.1f} px")
        print(f"    Distance to Pt: {dist_mm:.1f} mm")
        print(f"    Scale Ratio: {px_per_mm:.2f} px/mm")
        print(f"    Linear Mapping: 0.05mm error -> ~{0.05 * px_per_mm:.2f} px reproj error")

if XA_3d is not None:
    check_scale_and_magnification(XA_3d, cams)

--- Scale & Magnification Analysis ---
  Cam 0:
    Focal Length: 9009.8 px
    Distance to Pt: 205.5 mm
    Scale Ratio: 43.83 px/mm
    Linear Mapping: 0.05mm error -> ~2.19 px reproj error
  Cam 1:
    Focal Length: 8994.7 px
    Distance to Pt: 214.6 mm
    Scale Ratio: 41.91 px/mm
    Linear Mapping: 0.05mm error -> ~2.10 px reproj error
  Cam 2:
    Focal Length: 9009.0 px
    Distance to Pt: 216.1 mm
    Scale Ratio: 41.70 px/mm
    Linear Mapping: 0.05mm error -> ~2.08 px reproj error


In [15]:
if dataset['frames']:
    test_fid = dataset['frames'][0]
    XA_3d, XB_3d = reconstruct_and_analyze(test_fid, dataset, cams)
    
    if XA_3d is not None and XB_3d is not None:
        L = np.linalg.norm(XA_3d - XB_3d)
        print(f"\nReconstructed Wand Length: {L:.4f} mm")

--- Frame 0 Reconstruction Analysis ---

  Point A:
    3D Position: [-0.94835303  1.33830623  7.09354285]
    Ray Errors (mm): ['0.0079', '0.0696', '0.0711']
    Mean Recon Error: 0.049537 mm

  Point B:
    3D Position: [4.24164427 6.48229889 0.30211409]
    Ray Errors (mm): ['0.0405', '0.0924', '0.0584']
    Mean Recon Error: 0.063755 mm

Reconstructed Wand Length: 9.9760 mm


In [7]:
import numpy as np
from collections import defaultdict
import openlpt as lpt
from modules.camera_calibration.wand_calibration.refractive_geometry import (
    build_pinplate_ray_cpp, triangulate_point, point_to_ray_dist
)

def validate_and_compute_reprojection_errors_full(dataset, cams, tolerance_mm=1e-4):
    """
    1. 初次重构 3D (X_orig)
    2. 投影 (X_orig -> uv_proj)
    3. 二次重构 (uv_proj -> X_recon)
    4. 验证: |X_orig - X_recon| < tol AND MeanRayErr(X_recon) < tol
    5. 统计通过验证的点的投影误差 |uv_obs - uv_proj|
    """
    
    print(f"=== Full Dataset Verification (Tolerance: {tolerance_mm} mm) ===")
    
    reproj_errors = defaultdict(list)
    valid_points_count = 0
    skipped_points_count = 0
    total_points_processed = 0
    
    # helper to process one point dict (obsA or obsB)
    def process_point_set(obs_map, fid, label):
        nonlocal valid_points_count, skipped_points_count
        
        if fid not in obs_map: 
            return
            
        obs_dict = obs_map[fid]
        if not obs_dict or len(obs_dict) < 2:
            return

        # --- Step 1: Initial Reconstruction (Observation -> 3D Orig) ---
        rays_orig = []
        active_cids = []
        for cid, uv in obs_dict.items():
            if cid in cams:
                ray = build_pinplate_ray_cpp(cams[cid], uv)
                if ray.valid:
                    rays_orig.append(ray)
                    active_cids.append(cid)
        
        if len(rays_orig) < 2: return

        X_orig, _, success, _ = triangulate_point(rays_orig)
        if not success: return

        # --- Step 2: Projection (3D Orig -> uv_proj) ---
        pt_world = lpt.Pt3D(float(X_orig[0]), float(X_orig[1]), float(X_orig[2]))
        uv_proj_map = {}
        
        for cid in active_cids:
            # 理论投影点
            uv_p = cams[cid].project(pt_world)
            uv_proj_map[cid] = np.array([uv_p[0], uv_p[1]])

        # --- Step 3: Re-Reconstruction (uv_proj -> 3D Recon) ---
        rays_recon = []
        for cid, uv_p in uv_proj_map.items():
            # 使用投影点建立射线
            ray = build_pinplate_ray_cpp(cams[cid], uv_p)
            if ray.valid:
                rays_recon.append(ray)
        
        if len(rays_recon) < 2: return # Should not happen if Step 1 passed
        
        X_recon, _, success_recon, _ = triangulate_point(rays_recon)
        if not success_recon: return
        
        # --- Step 4: Validation Checks ---
        # A. Euclidean Consistency Error (Orig vs Recon)
        dist_consistency = np.linalg.norm(X_orig - X_recon)
        
        # B. Reconstruction Ray Error (of X_recon against theoretical rays)
        #    如果模型完美可逆，这个值应该极小
        recon_ray_dists = [point_to_ray_dist(X_recon, r.o, r.d) for r in rays_recon]
        mean_recon_err = np.mean(recon_ray_dists)
        
        # 判断是否有效
        if dist_consistency > tolerance_mm or mean_recon_err > tolerance_mm:
            # print(f"  [Skip Frame {fid} {label}] Failed check: Diff={dist_consistency:.4e}, ReconErr={mean_recon_err:.4e}")
            skipped_points_count += 1
            return
        
        # --- Step 5: Compute Reprojection Error (For Valid Points) ---
        # Error = |uv_obs - uv_proj|
        valid_points_count += 1
        
        for cid in active_cids:
            uv_obs = np.array(obs_dict[cid])
            uv_proj = uv_proj_map[cid]
            
            err_px = np.linalg.norm(uv_obs - uv_proj)
            reproj_errors[cid].append(err_px)

    # --- Main Loop ---
    frames = dataset['frames']
    for i, fid in enumerate(frames):
        process_point_set(dataset['obsA'], fid, "PtA")
        process_point_set(dataset['obsB'], fid, "PtB")
        total_points_processed += 1
        
        if i % 100 == 0:
            print(f"  Processed {i}/{len(frames)} frames...")

    # --- Statistics Output ---
    print(f"\n{'='*50}")
    print(f"Validation Summary")
    print(f"{'='*50}")
    print(f"Total Points Checked: {total_points_processed * 2} (approx)")
    print(f"Valid Points (Passed {tolerance_mm}mm Tol): {valid_points_count}")
    print(f"Skipped Points (Failed Consistency): {skipped_points_count}")
    print(f"Pass Rate: {valid_points_count / (valid_points_count + skipped_points_count + 1e-9) * 100:.2f}%")
    print(f"{'-'*50}")
    print(f"{'Cam ID':<8} | {'Mean (px)':<10} | {'Std (px)':<10} | {'Max (px)':<10} | {'Count':<8}")
    print(f"{'-'*50}")
    
    total_mean_errs = []
    
    sorted_cam_ids = sorted(reproj_errors.keys())
    for cid in sorted_cam_ids:
        errs = np.array(reproj_errors[cid])
        if len(errs) > 0:
            mean_e = np.mean(errs)
            std_e = np.std(errs)
            max_e = np.max(errs)
            count = len(errs)
            total_mean_errs.append(mean_e)
            
            print(f"{cid:<8} | {mean_e:<10.4f} | {std_e:<10.4f} | {max_e:<10.4f} | {count:<8}")
        else:
            print(f"{cid:<8} | {'N/A':<10} | {'N/A':<10} | {'N/A':<10} | {0:<8}")
            
    print(f"{'-'*50}")
    if total_mean_errs:
        print(f"Overall Mean Reprojection Error: {np.mean(total_mean_errs):.4f} px")

# Run It
validate_and_compute_reprojection_errors_full(dataset, cams, tolerance_mm=1e-4)

=== Full Dataset Verification (Tolerance: 0.0001 mm) ===
  Processed 0/2331 frames...
  Processed 100/2331 frames...
  Processed 200/2331 frames...
  Processed 300/2331 frames...
  Processed 400/2331 frames...
  Processed 500/2331 frames...
  Processed 600/2331 frames...
  Processed 700/2331 frames...
  Processed 800/2331 frames...
  Processed 900/2331 frames...
  Processed 1000/2331 frames...
  Processed 1100/2331 frames...
  Processed 1200/2331 frames...
  Processed 1300/2331 frames...
  Processed 1400/2331 frames...
  Processed 1500/2331 frames...
  Processed 1600/2331 frames...
  Processed 1700/2331 frames...
  Processed 1800/2331 frames...
  Processed 1900/2331 frames...
  Processed 2000/2331 frames...
  Processed 2100/2331 frames...
  Processed 2200/2331 frames...
  Processed 2300/2331 frames...

Validation Summary
Total Points Checked: 4662 (approx)
Valid Points (Passed 0.0001mm Tol): 4607
Skipped Points (Failed Consistency): 55
Pass Rate: 98.82%
--------------------------------